# Assignment 1: Brain Tumor MRI Classification
**ML for Health 2026**

Dataset: [Kaggle Brain Tumor MRI Classification](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)  
Classes: `glioma` | `meningioma` | `pituitary` | `no_tumor`

---

## Instructions

- Implement the functions in `assignment.py` by following the TODOs.
- Use this notebook for exploration, visualization, and answering the written questions.
- Run `pytest tests/ -v` in the terminal to check your implementation.
- Push to `main` when you are ready to submit.
- Report any issues in `experiences.md`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix

from assignment import (
    set_seed,
    compute_metrics,
    compute_class_weights,
    build_dataloaders,
    SimpleCNN,
    build_resnet18,
    run_epoch,
)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

---
## Exercise 1: Task Framing  *(written questions)*

**1a.** Formulate the supervised learning objective: define input $x$, output $y$, and the prediction task.

> *Your answer here.*

**1b.** Propose one clinically meaningful **primary metric** and one **secondary metric**. Briefly justify each.

> *Your answer here.*

**1c.** Explain why plain accuracy can be misleading when classes are imbalanced.

> *Your answer here.*

**1d.** Name one potential source of data leakage in medical image classification.

> *Your answer here.*

---
## Exercise 3: Compute Metrics  *(implemented in assignment.py)*

For class **glioma** in a one-vs-rest setting:
$$TP = 42, \quad FP = 8, \quad FN = 18$$

Implement `compute_metrics(tp, fp, fn)` in `assignment.py`, then run the cell below.

In [ ]:
precision, recall, f1 = compute_metrics(tp=42, fp=8, fn=18)
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1-score  : {f1:.4f}')
print()
print(f'Interpretation: {recall*100:.0f}% of true glioma cases are detected.')
print(f'{(1-recall)*100:.0f}% of glioma cases are missed (false negatives).')

---
## Exercise 4: Data Splits and Leakage  *(written questions)*

You have MRI slices from 250 patients. Each patient has multiple images.

**4a.** Why is a random image-level split risky here?

> *Your answer here.*

**4b.** Propose a patient-level train/validation/test split ratio.

> *Your answer here.*

**4c.** Should data augmentation be applied to training, validation, or test?

> *Your answer here.*

**4d.** Why should the held-out test set remain untouched until the very end?

> *Your answer here.*

---
## Exercise 5: Class Imbalance  *(implemented in assignment.py)*

Training class counts:
$$\text{glioma}=120,\ \text{meningioma}=240,\ \text{pituitary}=300,\ \text{no\_tumor}=840$$

Implement `compute_class_weights(class_counts)` in `assignment.py`, then run the cell below.

In [ ]:
counts = {"glioma": 120, "meningioma": 240, "pituitary": 300, "no_tumor": 840}
weights = compute_class_weights(counts)
for cls, w in weights.items():
    print(f'{cls:>12}: {w:.4f}')

---
## Exercise 2: Build and Train a Classifier  *(implemented in assignment.py)*

### 2a. Load the dataset

Download the dataset from Kaggle and place it at:
```
data/
  Training/
    glioma/
    meningioma/
    pituitary/
    no_tumor/
  Testing/
    ...
```

In [ ]:
data_root = Path('data')
train_loader, val_loader, test_loader = build_dataloaders(
    train_root=data_root / 'Training',
    test_root=data_root / 'Testing',
    val_fraction=0.2,
    batch_size=32,
    seed=42,
)
print('Train batches:', len(train_loader))
print('Val   batches:', len(val_loader))
print('Test  batches:', len(test_loader))

### 2b. Train SimpleCNN

In [ ]:
model = SimpleCNN(num_classes=4).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10
history = []

for epoch in range(num_epochs):
    train_loss, train_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_targets, val_preds = run_epoch(model, val_loader, criterion, None, device)
    stats = dict(epoch=epoch+1, train_loss=train_loss, train_acc=train_acc,
                 val_loss=val_loss, val_acc=val_acc)
    history.append(stats)
    print(stats)

In [ ]:
# Plot training curves
epochs = [h['epoch'] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(epochs, [h['train_loss'] for h in history], label='train')
axes[0].plot(epochs, [h['val_loss'] for h in history], label='val')
axes[0].set(title='Loss', xlabel='Epoch', ylabel='Loss')
axes[0].legend()
axes[1].plot(epochs, [h['train_acc'] for h in history], label='train')
axes[1].plot(epochs, [h['val_acc'] for h in history], label='val')
axes[1].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Validation report
class_names = ['glioma', 'meningioma', 'no_tumor', 'pituitary']
_, _, val_targets, val_preds = run_epoch(model, val_loader, criterion, None, device)
print(classification_report(val_targets, val_preds, target_names=class_names))

cm = confusion_matrix(val_targets, val_preds)
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title('Validation Confusion Matrix -- SimpleCNN')
plt.show()

### 2c. Train pretrained ResNet18 and compare

In [ ]:
resnet = build_resnet18(num_classes=4).to(device)
optimizer_r = torch.optim.Adam(resnet.parameters(), lr=1e-5)

for epoch in range(10):
    train_loss, train_acc, _, _ = run_epoch(resnet, train_loader, criterion, optimizer_r, device)
    val_loss, val_acc, _, _ = run_epoch(resnet, val_loader, criterion, None, device)
    print(f'Epoch {epoch+1}: train_acc={train_acc:.3f}  val_acc={val_acc:.3f}')

---
## Exercise 6: Model Comparison  *(written questions)*

Two models evaluated on the same held-out test set:

| Model | Accuracy | Macro-F1 | Worst-class Recall |
|-------|----------|----------|--------------------|
| CNN-A | 0.93     | 0.79     | 0.61               |
| CNN-B | 0.91     | 0.84     | 0.74               |

**6a.** Which model is better if clinical risk from missed tumor cases is the top priority?

> *Your answer here.*

**6b.** Which metric most directly supports that decision?

> *Your answer here.*

**6c.** What extra checks would you run before finalizing your choice?

> *Your answer here.*

---
## Discussion: Why this is not a clinical-grade benchmark

Reflect on at least **two limitations** of the setup used in this assignment from a clinical deployment perspective.

> *Your answer here.*